In [17]:
#=============================================================================
#  Global Supply Chain Disruption Predictor
#  by Grace Chung

# Purpose: Machine learning model that can identify top risk factors 
# and predict future shiptments
#=============================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

# --- Configuration ---
DATA_PATH   = "global_supply_chain_risk_2026.csv"
TARGET_COL  = "Disruption_Occurred"
TEST_SIZE   = 0.20
RANDOM_SEED = 42


# --- 1. Load data ---
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
    print(f"Disruption rate: {df[TARGET_COL].mean():.1%}")
    return df


# --- 2. Feature engineering ---
WEATHER_SEVERITY = {"Clear": 0, "Rain": 1, "Fog": 2, "Storm": 3, "Hurricane": 4}

CATEGORICAL_COLS = [
    "Transport_Mode",
    "Product_Category",
    "Origin_Port",
    "Destination_Port",
]

NUMERIC_FEATURES = [
    "Distance_km",
    "Weight_MT",
    "Fuel_Price_Index",
    "Geopolitical_Risk_Score",
    "Carrier_Reliability_Score",
    "Lead_Time_Days",
]

ENGINEERED_FEATURES = [
    "Weather_Severity",
    "Month",
    "Quarter",
    "Risk_Score",
    "Is_High_Risk_Weather",
    "Is_Long_Lead_Time",
]


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Weather severity ordinal
    df["Weather_Severity"] = df["Weather_Condition"].map(WEATHER_SEVERITY)

    # Temporal features
    df["Date"]    = pd.to_datetime(df["Date"])
    df["Month"]   = df["Date"].dt.month
    df["Quarter"] = df["Date"].dt.quarter

    # Composite risk score (weighted combination of top predictors)
    df["Risk_Score"] = (
        df["Geopolitical_Risk_Score"] * 0.4
        + df["Weather_Severity"] * 0.3
        + (1 - df["Carrier_Reliability_Score"]) * 10 * 0.3
    )

    # Binary flags
    df["Is_High_Risk_Weather"] = (df["Weather_Severity"] >= 3).astype(int)
    df["Is_Long_Lead_Time"]    = (
        df["Lead_Time_Days"] > df["Lead_Time_Days"].quantile(0.75)
    ).astype(int)

    # Label-encode categorical columns
    le = LabelEncoder()
    for col in CATEGORICAL_COLS:
        df[f"{col}_Enc"] = le.fit_transform(df[col])

    return df


def get_feature_list() -> list[str]:
    enc_features = [f"{c}_Enc" for c in CATEGORICAL_COLS]
    return NUMERIC_FEATURES + ENGINEERED_FEATURES + enc_features


# --- 3. Build models---
def build_models() -> dict:
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
        ]),
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=5,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            random_state=RANDOM_SEED,
        ),
    }


# --- 4. Evaluate ---
def cross_validate_all(models: dict, X_train, y_train, cv_folds: int = 5) -> dict:
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_SEED)
    results = {}
    print(f"\n── {cv_folds}-fold Cross-Validation (ROC-AUC) ─────────────────────────")
    for name, model in models.items():
        scores = cross_val_score(model, X_train, y_train, cv=cv,
                                 scoring="roc_auc", n_jobs=-1)
        results[name] = {"mean": scores.mean(), "std": scores.std(), "scores": scores}
        print(f"  {name:<25}  AUC = {scores.mean():.4f} ± {scores.std():.4f}")
    return results


def evaluate_best_model(model, X_test, y_test) -> dict:
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc    = roc_auc_score(y_test, y_prob)
    ap     = average_precision_score(y_test, y_prob)

    print("\n── Test-set performance ───────────────────────────────────────────────")
    print(classification_report(y_test, y_pred,
                                target_names=["No Disruption", "Disruption"]))
    print(f"ROC-AUC       : {auc:.4f}")
    print(f"Avg Precision : {ap:.4f}")
    print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

    return {"auc": auc, "avg_precision": ap,
            "y_pred": y_pred, "y_prob": y_prob}


def print_feature_importance(model, features: list[str]) -> None:
    # Works for tree models; falls back to logistic coefficients
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    else:
        importances = np.abs(model.named_steps["clf"].coef_[0])

    imp_series = pd.Series(importances, index=features).sort_values(ascending=False)
    print("\n── Top 10 Feature Importances ────────────────────────────────────────")
    for feat, val in imp_series.head(10).items():
        bar = "█" * int(val * 150)
        print(f"  {feat:<35}  {val:.4f}  {bar}")


# --- 5. Predict on new shipments ---
def predict_shipment(model, df_ref: pd.DataFrame, **kwargs) -> dict:
    """
    Predict disruption probability for a single shipment.

    Example
    -------
    predict_shipment(
        best_model, df,
        Distance_km=9000,
        Weight_MT=300,
        Fuel_Price_Index=2.5,
        Geopolitical_Risk_Score=7.0,
        Carrier_Reliability_Score=0.6,
        Lead_Time_Days=45,
        Weather_Condition="Storm",
        Transport_Mode="Sea",
        Product_Category="Electronics",
        Origin_Port="Shanghai",
        Destination_Port="Rotterdam",
        Date="2026-06-15",
    )
    """
    row = {col: [kwargs.get(col, df_ref[col].median()
                 if df_ref[col].dtype != object
                 else df_ref[col].mode()[0])]
           for col in df_ref.columns
           if col not in (TARGET_COL, "Shipment_ID")}
    row_df = pd.DataFrame(row)
    row_df = engineer_features(row_df)
    X_new  = row_df[get_feature_list()]
    prob   = model.predict_proba(X_new)[0, 1]
    label  = "DISRUPTION" if prob >= 0.5 else "NO DISRUPTION"
    return {"probability": round(prob, 4), "prediction": label}


# --- Main ---
def main():
    # 1. Load
    df = load_data(DATA_PATH)

    # 2. Engineer features
    df = engineer_features(df)
    features = get_feature_list()

    X = df[features]
    y = df[TARGET_COL]

    # 3. Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )
    print(f"\nTrain: {len(X_train):,}  |  Test: {len(X_test):,}")

    # 4. Cross-validate
    models  = build_models()
    cv_results = cross_validate_all(models, X_train, y_train)

    # 5. Pick best & evaluate
    best_name  = max(cv_results, key=lambda k: cv_results[k]["mean"])
    best_model = models[best_name]
    print(f"\n Best model: {best_name}")
    best_model.fit(X_train, y_train)
    evaluate_best_model(best_model, X_test, y_test)

    # 6. Feature importance
    print_feature_importance(best_model, features)

if __name__ == "__main__":
    main()


Loaded 5,000 rows × 14 columns
Disruption rate: 61.3%

Train: 4,000  |  Test: 1,000

── 5-fold Cross-Validation (ROC-AUC) ─────────────────────────
  Logistic Regression        AUC = 0.8228 ± 0.0172
  Random Forest              AUC = 0.8198 ± 0.0153
  Gradient Boosting          AUC = 0.8097 ± 0.0164

★ Best model: Logistic Regression

── Test-set performance ───────────────────────────────────────────────
               precision    recall  f1-score   support

No Disruption       0.67      0.64      0.66       387
   Disruption       0.78      0.80      0.79       613

     accuracy                           0.74      1000
    macro avg       0.73      0.72      0.72      1000
 weighted avg       0.74      0.74      0.74      1000

ROC-AUC       : 0.8233
Avg Precision : 0.8953

Confusion Matrix:
[[249 138]
 [122 491]]

── Top 10 Feature Importances ────────────────────────────────────────
  Lead_Time_Days                       0.7601  ███████████████████████████████████████████████████